<a href="https://colab.research.google.com/github/kanakamvasundhara/research-paper-rag/blob/main/research_paper_qa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. Install required packages
!pip install pypdf faiss-cpu sentence-transformers transformers

# 2. Import libraries
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import pipeline

# 3. Load and Extract PDF text
pdf_file = "/content/nlp_application.pdf"
reader = PdfReader(pdf_file)
pages = []

for page in reader.pages:
    text = page.extract_text()
    if text:
        pages.append(text)

print("Number of pages extracted:", len(pages))

# 4. Chunk the extracted text
chunks = []
chunk_size = 500
overlap = 100

for page_no, text in enumerate(pages, start=1):
    words = text.split()
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        if len(chunk.strip()) > 50:
            chunks.append({
                "text": chunk,
                "page": page_no
            })

print("Total chunks created:", len(chunks))

# 5. Generate text embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

# 6. Store embeddings in FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype("float32"))
print("Vectors stored in FAISS:", index.ntotal)

# 7. Initialize the text2text generator (Fixed pipeline for FLAN-T5)
generator = pipeline(
    "text-generation", # Changed from "text2text-generation" to "text-generation"
    model="google/flan-t5-base",
    max_new_tokens=250
)

# 8. Define the QA Function (Fixed syntax loop and optimized context size)
def ask_question(question, k=2):
    # Encode user question
    query_embedding = model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    # Search FAISS index for closest matching chunks
    distances, indices = index.search(query_embedding, k)
    retrieved = []
    for idx in indices[0]:
        if idx < len(chunks):
            retrieved.append(chunks[idx])

    # Construct the RAG prompt context
    context = "\n\n".join(
        f"[Page {item['page']}]\n{item['text']}"
        for item in retrieved
    )

    prompt = f"""
Answer the question using ONLY the information given in the context.
If the answer is not available in the context, say: "Information not found in the research paper."

Give a short and clear answer.

Context:
{context}

Question:
{question}

Answer:
"""

    # Generate response from FLAN-T5
    response = generator(prompt)[0]["generated_text"]

    print("\nANSWER:")
    print(response)

    print("\nSOURCES:")
    for item in retrieved:
        print(f"- Page {item['page']}")

    return response


Number of pages extracted: 15
Total chunks created: 24


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CohereCompassForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2

In [3]:
ask_question("What is the main finding of the paper?")


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1274 > 512). Running this sequence through the model will result in indexing errors
[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER:

Answer the question using ONLY the information given in the context.
If the answer is not available in the context, say: "Information not found in the research paper."

Give a short and clear answer.

Context:
[Page 12]
[25] Mitchell P Marcus, Mary Ann Marcinkiewicz, and Beatrice Santorini. Building a large annotated corpus of english: The penn treebank. Computational linguistics, 19(2):313–330, 1993. [26] David McClosky, Eugene Charniak, and Mark Johnson. Effective self-training for parsing. In Proceedings of the Human Language Technology Conference of the NAACL, Main Conference , pages 152–159. ACL, June 2006. [27] Ankur Parikh, Oscar Täckström, Dipanjan Das, and Jakob Uszkoreit. A decomposable attention model. In Empirical Methods in Natural Language Processing , 2016. [28] Romain Paulus, Caiming Xiong, and Richard Socher. A deep reinforced model for abstractive summarization. arXiv preprint arXiv:1705.04304, 2017. [29] Slav Petrov, Leon Barrett, Romain Thibaux, and Dan Kl

'\nAnswer the question using ONLY the information given in the context.\nIf the answer is not available in the context, say: "Information not found in the research paper."\n\nGive a short and clear answer.\n\nContext:\n[Page 12]\n[25] Mitchell P Marcus, Mary Ann Marcinkiewicz, and Beatrice Santorini. Building a large annotated corpus of english: The penn treebank. Computational linguistics, 19(2):313–330, 1993. [26] David McClosky, Eugene Charniak, and Mark Johnson. Effective self-training for parsing. In Proceedings of the Human Language Technology Conference of the NAACL, Main Conference , pages 152–159. ACL, June 2006. [27] Ankur Parikh, Oscar Täckström, Dipanjan Das, and Jakob Uszkoreit. A decomposable attention model. In Empirical Methods in Natural Language Processing , 2016. [28] Romain Paulus, Caiming Xiong, and Richard Socher. A deep reinforced model for abstractive summarization. arXiv preprint arXiv:1705.04304, 2017. [29] Slav Petrov, Leon Barrett, Romain Thibaux, and Dan Kl